In [ ]:
# Q1. Embedding a query

from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"

v = embed.encode(q1)

In [31]:
print(len(v))
print(v[0])

384
-0.02058203437252893


In [ ]:
# Q2. Cosine similarity

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [34]:
page = next(
    (
        doc for doc in documents
        if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
    ),
    None
)

if page is None:
    print("Page not found")
else:
    dv = embed.encode(page["content"])
    cosine_similarity = v.dot(dv)
    print(cosine_similarity)

0.36107027225589694


In [ ]:
# Q3. Chunking and search by hand

import numpy as np

from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)


chunk_texts = [chunk["content"] for chunk in chunks]
dvChunks = embed.encode_batch(chunk_texts)

X = np.array(dvChunks)

scores = X.dot(v)

best_idx = np.argmax(scores)

print(chunks[best_idx]["filename"])
# print(chunks[best_idx]["content"])
print(scores[best_idx])

02-vector-search/lessons/07-sqlitesearch-vector.md
0.6489017718578813


In [ ]:
# Q4. Vector search with minsearch

from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["content"])
vindex.fit(dvChunks, chunks)

q2 = "What metric do we use to evaluate a search engine?"
v2 = embed.encode(q2)

results = vindex.search(v2, num_results=1)
print(results[0]["filename"])


04-evaluation/lessons/05-search-metrics.md


In [39]:
# Q5. Text search vs vector search

q3 = "How do I store vectors in PostgreSQL?"
v3 = embed.encode(q3)

results = vindex.search(v3, num_results=5)
for result in results:
    print(result["filename"])

02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


In [41]:
from minsearch import Index

tindex = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
tindex.fit(chunks)

results = tindex.search(q3, num_results=5)
for result in results:
    print(result["filename"])

02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [42]:
# Q6. Hybrid search

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

q4 = "How do I give the model access to tools?"
v4 = embed.encode(q4)
resultsVectorSearch = vindex.search(v4)

resultsTextSearch = tindex.search(q4)

resultsHybridSearch = rrf([resultsVectorSearch, resultsTextSearch])
print(resultsHybridSearch[0]["filename"])

01-agentic-rag/lessons/13-function-calling.md
